# Open Problem 4.1 — ODE Gradient Flow (scipy RK45, Google Drive)

**Conjecture:** $\lim_{m\to\infty} C(m,f^*) = \#\{x\in[-1,1]:f^{*\prime\prime}(x)=0\text{ and changes sign}\}$

Uses **scipy `solve_ivp` (adaptive RK45)** — identical to `simulate_parallel.py`.  
Adaptive step sizing is required: at large $m$, the initial network output is $O(\sqrt{m})$,
creating a fast transient where fixed-step RK4 with $dt=0.01$ fails completely for $m\ge1000$.

All outputs are written to **Google Drive** so runs survive session restarts.  
Re-run any cell — completed runs are skipped automatically via `run_meta.csv`.

**Setup:** No GPU needed — set runtime to **CPU** (saves T4 quota with zero speed penalty).  
**After the run:** find results in your Drive at `NSF_RTG_ODE/`.  
Copy that folder into `figures/Replication data/` locally for post-processing scripts.

In [ ]:
# ── 1. Mount Drive ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import platform
print(f'Python : {platform.python_version()}')
print('Solver : scipy RK45 (adaptive) — no GPU required.')
print('Tip    : Runtime → Change runtime type → CPU  (saves GPU quota).')

In [ ]:
# ── 2. Imports ────────────────────────────────────────────────────────────────
try:
    from tqdm.notebook import tqdm
except ImportError:
    %pip install tqdm -q
    from tqdm.notebook import tqdm

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import os, csv, time, zipfile
from datetime import datetime

## Configuration
Edit `OUTPUT_DIR` to choose where in your Drive results are saved.  
Trim `M_VALUES` if you want a faster partial run — the job order ensures every target gets small-$m$ data first, so you can stop and resume at any time.

> **Re-running after a failed run?** Delete (or rename) the old `NSF_RTG_ODE/` folder in Drive first, or set `FORCE_RERUN = True` below so the notebook ignores the stale cached results.

In [ ]:
# ── 3. Configuration ──────────────────────────────────────────────────────────
T_FINAL  = 500
M_VALUES = [50, 100, 250, 500, 1000, 2000, 3500, 5000]

# Google Drive output folder
OUTPUT_DIR = '/content/drive/MyDrive/NSF_RTG_ODE'

# Set True to ignore existing run_meta.csv files and rerun everything.
# NOTE: runs completed with the old PyTorch RK4 solver are automatically rerun
#       even with FORCE_RERUN=False (they lack the 'solver' field in run_meta.csv).
FORCE_RERUN = False

# scipy solve_ivp parameters — matches simulate_parallel.py exactly.
# Adaptive RK45 is mandatory: at m=1000 the initial network output is O(sqrt(m)),
# and the fast early transient requires dt ~ 1/m that fixed-step RK4 cannot handle.
RTOL     = 1e-4
ATOL     = 1e-6
MAX_STEP = 1.0     # upper bound; scipy uses much smaller steps during t ~ 0–5

N_QUAD           = 200
N_SAVE           = 300       # trajectory snapshots stored per run
SEED             = 42
ACTIVE_THRESHOLD = 0.05
CLUSTER_TOL      = 0.02

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Output dir  : {OUTPUT_DIR}')
print(f'T_FINAL     : {T_FINAL}   solver=RK45  rtol={RTOL}  atol={ATOL}  max_step={MAX_STEP}')
print(f'M_VALUES    : {M_VALUES}')
print(f'Total jobs  : {len(M_VALUES) * 9}  ({len(M_VALUES)} m × 9 targets)')
print(f'FORCE_RERUN : {FORCE_RERUN}')

## Core math — numpy + scipy solve_ivp

In [ ]:
# ── 4. Quadrature grid + helpers ──────────────────────────────────────────────
from scipy.integrate import solve_ivp

X_NP  = np.linspace(-1.0, 1.0, N_QUAD)
DX_NP = float(X_NP[1] - X_NP[0])

def count_clusters(b_np):
    s = np.sort(b_np)
    return 1 + int(np.sum(np.diff(s) > CLUSTER_TOL))

def get_cluster_centers(b_np):
    s = np.sort(b_np)
    centers, group = [], [s[0]]
    for i in range(1, len(s)):
        if s[i] - s[i-1] > CLUSTER_TOL:
            centers.append(float(np.mean(group)))
            group = []
        group.append(s[i])
    centers.append(float(np.mean(group)))
    return np.array(centers)

print('Quadrature grid and helpers ready  (numpy + scipy).')

In [ ]:
# ── 5. Target functions and inflection points ─────────────────────────────────
TARGETS = {
    'sin_1pi': (r'$\sin(\pi x)$',      1,  lambda x: np.sin(    np.pi * x)),
    'x_cubed': (r'$x^3$',              1,  lambda x: x**3),
    'sin_2pi': (r'$\sin(2\pi x)$',     3,  lambda x: np.sin(2 * np.pi * x)),
    'poly_k3': (r'$x^5-3x^3$',         3,  lambda x: x**5 - 3*x**3),
    'sin_3pi': (r'$\sin(3\pi x)$',     5,  lambda x: np.sin(3 * np.pi * x)),
    'sin_4pi': (r'$\sin(4\pi x)$',     7,  lambda x: np.sin(4 * np.pi * x)),
    'sin_5pi': (r'$\sin(5\pi x)$',     9,  lambda x: np.sin(5 * np.pi * x)),
    'sin_6pi': (r'$\sin(6\pi x)$',    11,  lambda x: np.sin(6 * np.pi * x)),
    'sin_7pi': (r'$\sin(7\pi x)$',    13,  lambda x: np.sin(7 * np.pi * x)),
}
INFLECTIONS = {
    'sin_1pi': [0.0],
    'x_cubed': [0.0],
    'sin_2pi': [-0.5, 0.0, 0.5],
    'poly_k3': [-np.sqrt(0.9), 0.0, np.sqrt(0.9)],
    'sin_3pi': [-2/3, -1/3, 0.0, 1/3, 2/3],
    'sin_4pi': [-0.75, -0.5, -0.25, 0.0, 0.25, 0.5, 0.75],
    'sin_5pi': [-0.8, -0.6, -0.4, -0.2, 0.0, 0.2, 0.4, 0.6, 0.8],
    'sin_6pi': [-5/6, -4/6, -3/6, -2/6, -1/6, 0.0, 1/6, 2/6, 3/6, 4/6, 5/6],
    'sin_7pi': [-6/7,-5/7,-4/7,-3/7,-2/7,-1/7, 0.0, 1/7, 2/7, 3/7, 4/7, 5/7, 6/7],
}
print(f'{len(TARGETS)} targets defined.')

## Single-run function
Skips automatically if all output files and `run_meta.csv` already exist in Drive.

In [ ]:
# ── 6. run_one ────────────────────────────────────────────────────────────────
META_FIELDS = ['target','m','T','k_true','n_clusters','n_active',
               'loss','max_da','max_db','active_leq_k',
               'timestamp','solver','elapsed_s','rtol']

def run_one(target_key, m):
    T             = T_FINAL
    target_label, k_true, f_star_fn = TARGETS[target_key]
    f_star_np     = f_star_fn(X_NP)           # float64 — scipy prefers float64
    infl_x        = np.array(INFLECTIONS.get(target_key, []))

    out_dir = os.path.join(OUTPUT_DIR, target_key, f'm={m}', f'T={T}')
    f_traj  = os.path.join(out_dir, 'slide93_reproduction.png')
    f_clust = os.path.join(out_dir, 'clusters_vs_inflections.png')
    f_ode   = os.path.join(out_dir, 'ode_verification.png')
    f_csv   = os.path.join(out_dir, 'convergence_check.csv')
    f_meta  = os.path.join(out_dir, 'run_meta.csv')

    # ── Skip if already done with scipy (old PyTorch runs are rerun automatically) ──
    if (not FORCE_RERUN and
            all(os.path.exists(p) for p in [f_traj, f_clust, f_ode, f_csv, f_meta])):
        with open(f_meta, newline='') as mf:
            row = next(csv.DictReader(mf))
        if row.get('solver', '') == 'scipy_RK45':
            return {**row,
                    'm': int(row['m']), 'T': int(row['T']),
                    'k_true': int(row['k_true']),
                    'n_clusters': int(row['n_clusters']),
                    'n_active':   int(row['n_active']),
                    'active_leq_k': int(row['active_leq_k']),
                    'loss':   float(row['loss']),
                    'max_da': float(row['max_da']),
                    'max_db': float(row['max_db']),
                    '_skipped': True}
        else:
            print(f'  NOTE: {target_key} m={m} used old PyTorch RK4 — rerunning with scipy.')

    os.makedirs(out_dir, exist_ok=True)
    t0 = time.time()

    # ── Initialise (matches simulate_parallel.py exactly) ─────────────────────
    np.random.seed(SEED)
    a0 = np.random.randn(m)          * 0.01   # float64
    b0 = np.random.uniform(-1.0, 1.0, m)      # float64

    # ── ODE RHS (pure numpy — scipy calls this adaptively) ────────────────────
    def ode_rhs(t, y):
        a, b  = y[:m], y[m:]
        H     = np.maximum(0.0, X_NP[:, None] - b[None, :])   # (N_QUAD, m)
        r     = H @ a - f_star_np                               # (N_QUAD,)
        da    = -(H.T @ r) * DX_NP                             # (m,)
        cr    = np.cumsum(r[::-1])[::-1] * DX_NP               # (N_QUAD,) integral from b_j to 1
        idx   = np.searchsorted(X_NP, b).clip(0, N_QUAD - 1)  # (m,)
        db    = a * cr[idx]                                     # (m,)
        return np.concatenate([da, db])

    t_eval = np.linspace(0.0, float(T), N_SAVE)

    sol = solve_ivp(
        ode_rhs,
        (0.0, float(T)),
        np.concatenate([a0, b0]),
        method       = 'RK45',
        rtol         = RTOL,
        atol         = ATOL,
        max_step     = MAX_STEP,
        t_eval       = t_eval,
        dense_output = False,
    )

    if not sol.success:
        print(f'  WARNING: solve_ivp did not converge for {target_key} m={m}: {sol.message}')

    # ── Extract state ──────────────────────────────────────────────────────────
    n_snaps    = sol.y.shape[1]
    a_traj     = sol.y[:m,  :]    # (m, n_snaps)
    b_traj_raw = sol.y[m:,  :]    # (m, n_snaps)
    t_log      = sol.t            # (n_snaps,)

    # Sorted bias trajectories for the trajectory plot: (n_snaps, m)
    b_traj = np.sort(b_traj_raw, axis=0).T

    # Loss at ~60 evenly-spaced snapshots (vectorised)
    snap_stride = max(1, n_snaps // 60)
    snap_idxs   = list(range(0, n_snaps, snap_stride))
    loss_log    = []
    t_loss      = []
    for si in snap_idxs:
        a_si = a_traj[:, si];  b_si = b_traj_raw[:, si]
        H_si = np.maximum(0.0, X_NP[:, None] - b_si[None, :])
        r_si = H_si @ a_si - f_star_np
        loss_log.append(0.5 * np.trapz(r_si**2, X_NP))
        t_loss.append(t_log[si])

    # ── Final state ───────────────────────────────────────────────────────────
    a_np = a_traj[:, -1].copy()
    b_np = b_traj_raw[:, -1].copy()

    H_fin  = np.maximum(0.0, X_NP[:, None] - b_np[None, :])   # (N_QUAD, m)
    r_fin  = H_fin @ a_np - f_star_np
    final_loss = 0.5 * np.trapz(r_fin**2, X_NP)

    da_np  = -(H_fin.T @ r_fin) * DX_NP
    cr_fin = np.cumsum(r_fin[::-1])[::-1] * DX_NP
    idx_fin = np.searchsorted(X_NP, b_np).clip(0, N_QUAD - 1)
    R_np   = cr_fin[idx_fin]
    db_np  = a_np * R_np

    n_clusters      = count_clusters(b_np)
    cluster_centers = get_cluster_centers(b_np)
    a_max           = max(float(np.abs(a_np).max()), 1e-12)
    is_active       = np.abs(a_np) > ACTIVE_THRESHOLD * a_max
    n_active        = int(is_active.sum())

    sort_idx   = np.argsort(b_np)
    b_s, a_s   = b_np[sort_idx],  a_np[sort_idx]
    da_s, db_s = da_np[sort_idx], db_np[sort_idx]
    R_s        = R_np[sort_idx]
    active_s   = is_active[sort_idx]

    # Vectorised network evaluation (fast at large m)
    x_plot  = np.linspace(-1.0, 1.0, 500)
    f_plot  = np.maximum(0.0, x_plot[:, None] - b_np[None, :]) @ a_np   # (500,)
    x_fine  = np.linspace(-1.0, 1.0, 1000)
    f_fine  = np.maximum(0.0, x_fine[:, None] - b_np[None, :]) @ a_np   # (1000,)

    # ── Figure 1: trajectories / fit / loss ───────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'target={target_label},  m={m},  T={T}'
                 f'  |  clusters={n_clusters},  k={k_true}', fontsize=12)
    ax = axes[0]
    alpha    = min(0.4, 20.0 / m)
    plot_max = min(m, 200)           # cap at 200 lines — faster for large m
    step_j   = max(1, m // plot_max)
    for j in range(0, m, step_j):
        ax.plot(t_log, b_traj[:, j], color='steelblue', alpha=alpha, lw=0.4)
    ax.set_xlabel('Time'); ax.set_ylabel('Bias')
    ax.set_title('Sorted Bias Trajectories'); ax.set_xlim([0, T])
    ax = axes[1]
    ax.plot(x_plot, f_star_fn(x_plot), 'k--', lw=2, label=f'Target {target_label}')
    ax.plot(x_plot, f_plot,            'r-',  lw=2, label='Network $f$')
    y0, y1 = ax.get_ylim(); tick_h = (y1 - y0) * 0.06
    ax.vlines(cluster_centers, -tick_h/2, tick_h/2, colors='steelblue', lw=2.5, zorder=5)
    ax.set_xlabel('$x$'); ax.set_title('Final Fit vs Target')
    ax.legend(fontsize=8); ax.set_xlim([-1, 1])
    ax = axes[2]
    if loss_log:
        ax.semilogy(t_loss, loss_log, color='darkorange', lw=1.5)
    ax.set_xlabel('Time'); ax.set_ylabel('Loss')
    ax.set_title(f'Loss (log)   final={final_loss:.2e}')
    ax.set_xlim([0, T]); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f_traj, bbox_inches='tight'); plt.close()

    # ── Figure 2: clusters vs inflection points ───────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 4))
    fig.suptitle(f'target={target_label},  m={m},  T={T}  '
                 f'|  clusters={n_clusters},  k={k_true}  (C=k:{n_clusters==k_true})',
                 fontsize=11)
    ax.plot(x_fine, f_star_fn(x_fine), 'k-', lw=2, label=f'Target {target_label}')
    ax.plot(x_fine, f_fine, 'r-', lw=1.5, label='Final network')
    for cx in cluster_centers: ax.axvline(cx, color='steelblue', alpha=0.5, lw=0.7)
    for ix in infl_x:          ax.axvline(ix, color='green',     alpha=0.7, lw=1.2, ls='--')
    h, _ = ax.get_legend_handles_labels()
    h += [Line2D([0],[0], color='steelblue', lw=1.5, label=f'Clusters ({n_clusters})'),
          Line2D([0],[0], color='green',     lw=1.5, ls='--', label=f'Inflections k={k_true}')]
    ax.legend(handles=h, fontsize=9); ax.set_xlabel('$x$')
    ax.set_title('Cluster Locations vs Inflection Points')
    plt.tight_layout()
    plt.savefig(f_clust, bbox_inches='tight'); plt.close()

    # ── Figure 3: stationarity check ──────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f'Stationarity -- target={target_label},  m={m},  T={T}\n'
                 f'active={n_active},  k={k_true},  active<=k:{n_active<=k_true}',
                 fontsize=10)
    axes[0].scatter(b_s, np.abs(da_s)+1e-15, c=np.abs(a_s), cmap='viridis', s=10)
    axes[0].set(xlabel='$b_j$', ylabel='$|da/dt|$', yscale='log',
                title='Amplitude velocity'); axes[0].grid(True, alpha=0.3)
    axes[1].scatter(b_s, np.abs(db_s)+1e-15, c=np.abs(a_s), cmap='viridis', s=10)
    axes[1].set(xlabel='$b_j$', ylabel='$|db/dt|$', yscale='log',
                title='Bias velocity'); axes[1].grid(True, alpha=0.3)
    sizes = np.clip(np.abs(a_s) / a_max * 80, 4, 80)
    sc = axes[2].scatter(b_s, R_s, c=np.abs(a_s), cmap='viridis', s=sizes)
    axes[2].scatter(b_s[active_s], R_s[active_s],
                    edgecolors='red', facecolors='none', s=60, lw=1.2,
                    label=f'Active ({n_active})')
    axes[2].axhline(0, color='k', lw=1, ls='--')
    plt.colorbar(sc, ax=axes[2], label='$|a_j|$')
    axes[2].set(xlabel='$b_j$', ylabel='$R_j$',
                title=f'Integrated residual  active={n_active}<=k={k_true}:{n_active<=k_true}')
    axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f_ode, bbox_inches='tight'); plt.close()

    # ── convergence_check.csv ─────────────────────────────────────────────────
    with open(f_csv, 'w', newline='') as cf:
        w = csv.writer(cf)
        w.writerow(['neuron_idx','b_j','a_j','da_dt','db_dt','R_j','is_active','R_near_zero'])
        for i in range(m):
            w.writerow([sort_idx[i], f'{b_s[i]:.6f}', f'{a_s[i]:.6f}',
                        f'{da_s[i]:.6e}', f'{db_s[i]:.6e}', f'{R_s[i]:.6e}',
                        int(active_s[i]), int(abs(R_s[i]) < 1e-3)])

    elapsed = round(time.time() - t0, 2)

    # ── run_meta.csv ──────────────────────────────────────────────────────────
    result = {
        'target':       target_key,
        'm':            m,
        'T':            T,
        'k_true':       k_true,
        'n_clusters':   n_clusters,
        'n_active':     n_active,
        'loss':         final_loss,
        'max_da':       float(np.abs(da_np).max()),
        'max_db':       float(np.abs(db_np).max()),
        'active_leq_k': int(n_active <= k_true),
        'timestamp':    datetime.now().isoformat(timespec='seconds'),
        'solver':       'scipy_RK45',
        'elapsed_s':    elapsed,
        'rtol':         RTOL,
    }
    with open(f_meta, 'w', newline='') as mf:
        w = csv.DictWriter(mf, fieldnames=META_FIELDS)
        w.writeheader(); w.writerow(result)

    result['_elapsed'] = elapsed
    return result

print('run_one() defined  (scipy RK45 adaptive solver).')

## Run simulation
Jobs run in order **(m, target)** — all 9 targets at m=50 before m=100, etc.  
If the session times out, just re-run this cell: completed runs are skipped instantly.  
The summary CSV is updated after **every run**, so progress is never lost.

In [ ]:
# ── 7. Run all jobs ───────────────────────────────────────────────────────────
SUMMARY_PATH = os.path.join(OUTPUT_DIR, 'run_summary_parallel.csv')
SUMMARY_META = ['target','m','T','k_true','n_clusters','n_active',
                'loss','max_da','max_db','active_leq_k',
                'timestamp','solver','elapsed_s','rtol']

# Load any previously completed scipy runs from the summary CSV
completed_runs = {}
if os.path.exists(SUMMARY_PATH):
    with open(SUMMARY_PATH, newline='') as f:
        for row in csv.DictReader(f):
            if row.get('solver', '') == 'scipy_RK45':   # skip old PyTorch-RK4 rows
                completed_runs[(row['target'], int(row['m']))] = row
    if completed_runs:
        print(f'Loaded {len(completed_runs)} previously completed scipy runs.')

jobs = sorted(
    [(t, m) for t in TARGETS for m in M_VALUES],
    key=lambda x: (x[1], x[0])   # m first, then target name
)

n_skip = 0; n_done = 0
t_wall = time.time()

for target_key, m in tqdm(jobs, desc='ODE jobs'):
    result = run_one(target_key, m)
    if result is None:
        continue

    was_skipped = result.pop('_skipped', False)
    elapsed_s   = result.pop('_elapsed', result.get('elapsed_s', '?'))

    # Update in-memory dict and re-write summary CSV after every run
    completed_runs[(result['target'], int(result['m']))] = result
    rows_to_write = sorted(completed_runs.values(),
                           key=lambda r: (r['target'], int(r['m'])))
    with open(SUMMARY_PATH, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=SUMMARY_META)
        w.writeheader()
        for r in rows_to_write:
            w.writerow({k: r.get(k, '') for k in SUMMARY_META})

    status = 'SKIP' if was_skipped else f'DONE [{elapsed_s}s]'
    n_skip += was_skipped; n_done += (not was_skipped)
    print(f'  {status:<18}  {result["target"]:<12}  m={result["m"]:<5}  '
          f'C={result["n_clusters"]:<4}  k={result["k_true"]}  '
          f'C=k:{result["n_clusters"]==result["k_true"]}  '
          f'loss={float(result["loss"]):.3e}')

print(f'\nDone. {n_done} new, {n_skip} skipped.  '
      f'Wall time: {(time.time()-t_wall)/60:.1f} min')
print(f'Summary: {SUMMARY_PATH}')

## Convergence plot

In [ ]:
# ── 8. Convergence plot ───────────────────────────────────────────────────────
final_rows = sorted(completed_runs.values(),
                    key=lambda r: (r['target'], int(r['m'])))

TARGET_ORDER = ['sin_1pi','x_cubed','sin_2pi','poly_k3','sin_3pi',
                'sin_4pi','sin_5pi','sin_6pi','sin_7pi']
present = [t for t in TARGET_ORDER if any(r['target']==t for r in final_rows)]

ncols = 3
nrows = -(-len(present) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5*nrows), squeeze=False)
axes_flat = axes.flatten()

for ax_i, tkey in enumerate(present):
    ax     = axes_flat[ax_i]
    k_true = TARGETS[tkey][1]
    pts    = sorted([r for r in final_rows if r['target']==tkey], key=lambda r: int(r['m']))
    if pts:
        ax.plot([int(r['m']) for r in pts],
                [int(r['n_clusters']) for r in pts],
                marker='o', color='steelblue', lw=1.5, ms=5, label=f'T={T_FINAL}')
    ax.axhline(k_true, color='crimson', lw=2, ls='--', label=f'k={k_true}')
    ax.set_xlabel('Width $m$'); ax.set_ylabel('$C(m,f^*)$')
    ax.set_title(TARGETS[tkey][0]); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

for ax_i in range(len(present), len(axes_flat)):
    axes_flat[ax_i].set_visible(False)

fig.suptitle(
    f'Open Problem 4.1 — $C(m,f^*) \\to k$ as $m \\to \\infty$  (ODE flow, T={T_FINAL})\n'
    f'Crimson dashed = analytical $k$   |   {len(final_rows)} completed runs',
    fontsize=13)
plt.tight_layout()

conv_path = os.path.join(OUTPUT_DIR, 'convergence_plot_parallel.png')
plt.savefig(conv_path, bbox_inches='tight', dpi=130)
plt.close()
print(f'Saved -> {conv_path}')

from IPython.display import Image
Image(conv_path)

## Download
Your results are already in Google Drive at **`NSF_RTG_ODE/`**.  
To download: open [drive.google.com](https://drive.google.com), right-click the folder → **Download**.  

Alternatively, run the cell below to create a zip and download it directly to your browser.

In [ ]:
# ── 9. Optional: zip and download directly ────────────────────────────────────
# Only needed if you prefer a single zip over downloading from Drive UI.
# Skipped automatically if OUTPUT_DIR is empty.

from google.colab import files as colab_files

zip_path = '/content/NSF_RTG_ODE.zip'
n_files  = 0

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk(OUTPUT_DIR):
        for fname in fnames:
            fpath   = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, os.path.dirname(OUTPUT_DIR))
            zf.write(fpath, arcname)
            n_files += 1

mb = os.path.getsize(zip_path) / 1e6
print(f'Zipped {n_files} files -> {zip_path}  ({mb:.1f} MB)')
print('Triggering download...')
colab_files.download(zip_path)